# Neural Network Model

We will be training three types of neural network models
1. Temporal only Neural network model (Baseline)
2. Temporal + Static Neural Network (Single Stage Model)
3. Temporal Neural Network on Trend, Static Random Forest on Residuals (Two Stage Models)

This notebook illustrates the training process for the one stage neural network model on all features (Type 2). This is the neural network version of the single stage model.

The network sees only the temporal features of the month a flat was approved in (the 177 stock, COE and macroeconomic features of `data/feature_config_temporal.py`, from `data/features/final/temporal_scaled.csv` (already scaled by the data pipeline, so the network does no scaling of its own)), so every flat of a month gets the same prediction: the level of rent in that month.

Some details on the model architecture:

* **Network:**  We use an encoder like structure, with reducing hidden layer sizes to do automatic feature selection. The reducing layer size acts like a dimensionality reducer, much like PCA, with the encoding mechanism learned from gradient descent instead of computed via eigenvector decomposition.

* **Architecture** nn.ModuleList(`177 -> 64 -> 16 -> 1`), LeakyReLU activation between layers.

* **Output** The raw monthly rent as the target (standardised inside the training loop). Final layer has no activation.

* **Training Parameters** Adam Optimiser with learning rate 1e-3, batch size 2048, random seed 0, at most 100 epochs with early stopping: training stops once the validation RMSE has not improved for 10 epochs and the weights of the best epoch are kept.

* **Sample Use:** Train on the `train` months of the final feature tables (2021-01..2024-03), with the `val` months (2024-04..2025-03) held out for early stopping and evaluation, then run inference on all rows. The model is fitted once and is not refitted on the validation months: the test rows have no rents, so the validation rows are the only held-out rents the models can be compared on.# 02 Feed-forward network on the static and temporal features (single stage): training and inference

The network of `01_temporal_ffn.ipynb` with the static features of the flat added to its input, everything else the same: `LeakyReLU`, hidden layers `[64, 16]`, a linear output, the raw monthly rent as the target
(standardised inside the training loop), Adam with learning rate 1e-3, batch size 2048, seed 0. One network predicts the rent directly from the flat **and** the month.

* **Inputs:** the static features of the flat (`data/feature_config_static.py` from `static_train_scaled.csv` / `static_test_scaled.csv`: the scaled numeric columns, the ordinal columns standardised on the train rows and the one-hot columns; 151 inputs)
  and the 177 temporal features of the month it was approved in (`temporal_scaled.csv`, scaled by the data pipeline, as in notebook 01): 328 inputs. Identifier columns (town, block, coordinates, ...) are not inputs.
* **Rows:** one feature row per rental (114,841 train rows, 35,158 validation rows, 50,000 test rows), so predictions are made for the original rows directly (`ROW_ID` = position in `data/original_dataset/{train,test}.csv`).
* **Early stopping:** notebook 01's 100 epochs overfit once the flats' own features are inputs (the held-out error rises after the first few epochs), so training stops once the validation RMSE has not improved for 10 epochs
  (at most 100) and the weights of the best epoch are kept. The stopping epoch is chosen on the validation rows, which therefore slightly flatter the held-out predictions.
* **One fit:** on the train months (2021-01..2024-03). It predicts the train rows (in-sample), the validation rows (held out), the test rows, and the rent of a fixed set of flats in every month.
  The growth of the market is read from **the same flats** predicted in every month: the flats of 2025-01..03, predicted as if approved in each month (the flats differ between months, so the monthly mean prediction mixes growth with a change in the flat mix). Their features are used, not their rents.
* **Outputs:** the model in `results/models/`, the training statistics in `results/tables/static_temporal_ffn/`, the predictions and the fixed-flat levels in `03_evaluation/predictions/`. Evaluation: `03_evaluation/evaluation.ipynb`.

In [1]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

import os, sys
from pathlib import Path
_here = Path(os.environ.get('RENT_PROJECT_ROOT') or Path.cwd()).resolve()
sys.path.insert(0, str(next(p for p in [_here, *_here.parents] if (p / 'data' / 'config.py').exists())))      # final_report: data/config.py holds every path
from data import config as cfg
sys.path.insert(0, str(cfg.MODELLING_DIR))     # 02_modelling: nn_utils.py is shared by the single-stage and two-stage notebooks
import nn_utils as nu

warnings.filterwarnings('ignore')
pd.set_option('display.width', 200)
NAME = 'static_temporal_ffn'
HIDDEN = [64, 16]
ACTIVATION = 'leaky_relu'
MAX_EPOCHS, PATIENCE, BATCH_SIZE, LEARNING_RATE, WEIGHT_DECAY, SEED = 100, 10, 2048, 1e-3, 0.0, 0     # as notebook 01, plus early stopping on the validation RMSE: stop after PATIENCE epochs without improvement
nu.use_results_dir(cfg.SINGLE_STAGE_NN_RESULTS_DIR)
nu.ensure_output_dirs(NAME)
fit_args = dict(hidden=HIDDEN, activation=ACTIVATION, batch_size=BATCH_SIZE, learning_rate=LEARNING_RATE, weight_decay=WEIGHT_DECAY, patience=PATIENCE)

## 1. Data

In [2]:
temporal = nu.load_temporal()
TEMPORAL = list(temporal.columns)
train, test, STATIC = nu.load_static()
fit, val = (train['sample_type'] == 'train').to_numpy(), (train['sample_type'] == 'val').to_numpy()
month, y = train['month'].to_numpy(), train[nu.LABEL].to_numpy()
X, X_test = train[STATIC].to_numpy(np.float32), test[STATIC].to_numpy(np.float32)

print(f'{len(STATIC)} static + {len(TEMPORAL)} temporal = {len(STATIC) + len(TEMPORAL)} inputs')
print(f'train rows {fit.sum():,} (months 0..{nu.TRAIN_LAST}), validation rows {val.sum():,} (months {nu.TRAIN_LAST + 1}..{nu.VALIDATION_LAST}), test rows {len(test):,} (months {test["month"].min()}..{test["month"].max()})')

151 static + 177 temporal = 328 inputs
train rows 114,841 (months 0..38), validation rows 35,158 (months 39..50), test rows 50,000 (months 50..66)


## 2. Fit on the train months: 2021-01..2024-03, with early stopping

The network is trained on the train rows only, for at most 100 epochs, and scored after every epoch on the validation rows (never trained on). Training stops once the validation RMSE has not improved for 10 epochs and the weights of the best epoch are kept: the only fit, never refitted on the validation rows.

In [3]:
model = nu.fit_ffn(temporal, month[fit], y[fit], X=X[fit], valid=(month[val], X[val], y[val]), epochs=MAX_EPOCHS, seed=SEED, **fit_args)
best, ran, curve = model['config']['best_epoch'], model['config']['epochs_run'], model['history']['valid_rmse_dollar']
print(f'early stopping: best validation epoch {best}, stopped after {ran} of {MAX_EPOCHS} epochs; validation RMSE epoch 1 ${curve.iloc[0]:,.0f}, epoch {best} ${curve.loc[best - 1]:,.0f}')
pred_val = nu.predict_ffn(model, temporal, month[val], X[val])
print(model['history'].iloc[sorted({0, best - 1, ran - 1})].round(1).to_string(index=False))
pred_train, pred_test = nu.predict_ffn(model, temporal, month[fit], X[fit]), nu.predict_ffn(model, temporal, test['month'].to_numpy(), X_test)
level = nu.fixed_flat_levels(model, temporal, X[np.isin(month, nu.Q1_2025)], range(nu.N_MONTHS))     # the flats of 2025-01..03 (features only), predicted for each month
print(level.loc[44:].round(0).rename(lambda m: nu.month_label(m)[0]).to_frame('level of the same flats').T.to_string())

early stopping: best validation epoch 17, stopped after 27 of 100 epochs; validation RMSE epoch 1 $556, epoch 17 $492
 epoch  train_mse_std  train_rmse_dollar  valid_rmse_dollar
     1            0.6              586.2              556.2
    17            0.4              483.5              492.0
    27            0.4              480.9              517.7


                         2024-09  2024-10  2024-11  2024-12  2025-01  2025-02  2025-03  2025-04  2025-05  2025-06  2025-07  2025-08  2025-09  2025-10  2025-11  2025-12  2026-01  2026-02  2026-03  2026-04  2026-05  2026-06  2026-07
level of the same flats   3171.0   3043.0   3020.0   3101.0   3152.0   3083.0   3200.0   3199.0   3332.0   3388.0   3196.0   3360.0   3237.0   3243.0   3031.0   3101.0   2985.0   3045.0   3318.0   3475.0   3306.0   3234.0   3146.0


## 3. Save the model, the training statistics and the predictions

In [4]:
paths = [nu.MODEL_DIR / f'{NAME}.pt']
nu.save_ffn(model, paths[0], static_features=STATIC, temporal_features=TEMPORAL)
paths += [nu.save_table(model['history'], NAME, 'epoch_losses.csv', index=False)]                          # every epoch run; the model keeps the weights of the best validation epoch
paths += [nu.save_predictions(NAME, 'train', train[fit], pred_train),
          nu.save_predictions(NAME, 'val', train[val], pred_val),
          nu.save_predictions(NAME, 'test', test, pred_test),
          nu.save_levels(NAME, level)]
print('saved:', *paths, sep='\n  ')

saved:
  /Users/keltonn/Desktop/Repo/school/CS5228-project/final_report/02_modelling/single_stage_models/neural_network/results/models/static_temporal_ffn.pt
  /Users/keltonn/Desktop/Repo/school/CS5228-project/final_report/02_modelling/single_stage_models/neural_network/results/tables/static_temporal_ffn/epoch_losses.csv
  /Users/keltonn/Desktop/Repo/school/CS5228-project/final_report/03_evaluation/predictions/train/static_temporal_ffn.csv
  /Users/keltonn/Desktop/Repo/school/CS5228-project/final_report/03_evaluation/predictions/val/static_temporal_ffn.csv
  /Users/keltonn/Desktop/Repo/school/CS5228-project/final_report/03_evaluation/predictions/test/static_temporal_ffn.csv
  /Users/keltonn/Desktop/Repo/school/CS5228-project/final_report/03_evaluation/predictions/levels/static_temporal_ffn.csv
